# 04b — Video Model (CNN + Temporal Transformer)

This notebook is a **drop-in replacement** for the LSTM in `04_video_cnn_lstm.ipynb`.

**What changes:** LSTM → Temporal Transformer Encoder  
**What stays the same:** Frame extraction, Dataset, DataLoader, training loop, evaluation

## Why Transformer instead of LSTM?

| | LSTM | Temporal Transformer |
|---|---|---|
| Processes frames | Sequentially (one by one) | In parallel (all at once) |
| Long-range dependencies | Struggles (vanishing gradient) | Handles easily (attention) |
| Frame 1 ↔ Frame 16 | Information decays over time | Direct attention connection |
| Speed | Slower (sequential) | Faster (parallel) |
| Interpretability | Black box memory | Attention weights visualizable |

**Key idea:** The Transformer's self-attention lets Frame 1 (player preparing) directly
attend to Frame 16 (player scoring) with no information decay — something LSTM
struggles with over long sequences.

> **Prerequisite:** Run `04_video_cnn_lstm.ipynb` first to extract frames.
> Frames must exist at `/content/ucf_frames/`

## 1. Mount Drive and load frames

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Frames should already exist from 04_video_cnn_lstm.ipynb
# If not, copy from Drive:
if not os.path.exists('/content/ucf_frames'):
    print('Copying frames from Drive...')
    os.system('cp -r /content/drive/MyDrive/ContentRecognition/ucf_frames /content/')
    print('Done!')
else:
    print('Frames already on local SSD.')

## 2. Imports

In [ ]:
import os, time, random, math
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 3. Config

In [ ]:
CLASS_NAMES = [
    'Basketball', 'Biking', 'Bowling', 'CliffDiving',
    'GolfSwing', 'HorseRiding', 'Skiing', 'Surfing',
    'TennisSwing', 'SkateBoarding'
]
NUM_CLASSES      = len(CLASS_NAMES)
FRAMES_PER_VIDEO = 16
BATCH_SIZE       = 8
EPOCHS           = 10

FRAMES_DIR  = '/content/ucf_frames'
BASE_DIR    = '/content/drive/MyDrive/ContentRecognition'
CKPT_DIR    = f'{BASE_DIR}/checkpoints/video'
RESULTS_DIR = f'{BASE_DIR}/results/video'

os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Config ready.')

## 4. Dataset and DataLoaders
Identical to `04_video_cnn_lstm.ipynb` — no changes here.

In [ ]:
class VideoFrameDataset(Dataset):
    """
    Loads 16 saved JPEG frames for each video.
    Returns (video_tensor, label) where video_tensor shape = (16, 3, 224, 224).
    """
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        frame_files = sorted(os.listdir(video_path))
        frames = []
        for fname in frame_files:
            img = Image.open(os.path.join(video_path, fname)).convert('RGB')
            if self.transform:
                img = self.transform(img)
            frames.append(img)
        return torch.stack(frames), label


# Stronger augmentation to prevent overfitting
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Build samples list
class_to_idx = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}
all_samples  = []
for cls in CLASS_NAMES:
    cls_path = os.path.join(FRAMES_DIR, cls)
    if not os.path.exists(cls_path):
        print(f'WARNING: missing {cls}')
        continue
    for vid in os.listdir(cls_path):
        vp = os.path.join(cls_path, vid)
        if os.path.isdir(vp) and len(os.listdir(vp)) == FRAMES_PER_VIDEO:
            all_samples.append((vp, class_to_idx[cls]))

random.seed(42)                          # same seed as 04 → same split
random.shuffle(all_samples)
split         = int(0.8 * len(all_samples))
train_samples = all_samples[:split]
val_samples   = all_samples[split:]

train_dataset = VideoFrameDataset(train_samples, transform=train_transform)
val_dataset   = VideoFrameDataset(val_samples,   transform=val_transform)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)

print(f'Total  : {len(all_samples)} videos')
print(f'Train  : {len(train_dataset)}')
print(f'Val    : {len(val_dataset)}')

## 5. Positional Encoding

Transformers have NO built-in sense of order — unlike LSTM which processes
frames one-by-one and implicitly knows Frame 1 comes before Frame 2.

We must explicitly inject position information before the Transformer sees the sequence.

**Formula:**
```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```
Each position gets a unique pattern of sins and cosines across dimensions.
Frame 0 has one pattern, Frame 1 has a slightly different one — the model
learns to read these patterns as position information.

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding.
    Adds position information to frame feature sequence
    before the Transformer encoder processes it.

    Input shape  : (batch, seq_len, d_model)  e.g. (8, 16, 512)
    Output shape : (batch, seq_len, d_model)  same — position info added
    """
    def __init__(self, d_model, max_len=16, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Build the positional encoding table once — shape (max_len, d_model)
        pe       = torch.zeros(max_len, d_model)          # (16, 512)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # (16, 1)

        # Compute the division term: 10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)  # even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # odd  indices

        # Register as buffer — saved with model but not a trainable parameter
        pe = pe.unsqueeze(0)   # (1, 16, 512) — broadcast over batch
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        # self.pe: (1, max_len, d_model) → auto-broadcast
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


print('PositionalEncoding defined.')

## 6. CNN + Temporal Transformer Model

**Architecture:**
```
Input  : (batch, 16, 3, 224, 224)
           ↓
ResNet50 CNN — per frame feature extraction
  (all layers frozen — pure feature extractor)
           ↓
Linear projection: 2048 → 512
  (bring CNN output to Transformer dimension)
           ↓
Positional Encoding added
  (inject frame order information)
           ↓
Transformer Encoder (4 layers, 8 heads)
  Each frame attends to ALL other frames simultaneously
  Frame 1 ↔ Frame 16 — direct connection, no decay
           ↓
Mean pooling across all 16 frame outputs
  (aggregate temporal information)
           ↓
Classifier MLP: 512 → 256 → 10
```

**Why mean pooling instead of taking last token (like LSTM)?**  
LSTM is sequential — the last hidden state has seen everything.  
Transformer processes all frames in parallel — there is no special
'last' frame. Mean pooling averages information from all 16 frames equally.

In [ ]:
class CNN_Transformer(nn.Module):
    """
    ResNet50 CNN (frozen) + Temporal Transformer Encoder
    for video action recognition.

    Input  : (batch, frames, C, H, W) = (8, 16, 3, 224, 224)
    Output : (batch, num_classes)     = (8, 10)

    Args:
        num_classes  : number of output classes (default 10)
        d_model      : Transformer internal dimension (default 512)
        nhead        : number of attention heads (default 8)
                       d_model must be divisible by nhead
        num_layers   : number of Transformer encoder layers (default 4)
        dim_feedforward: hidden size of Transformer FFN (default 1024)
        dropout      : dropout rate inside Transformer (default 0.3)
    """
    def __init__(
        self,
        num_classes     = 10,
        d_model         = 512,
        nhead           = 8,
        num_layers      = 4,
        dim_feedforward = 1024,
        dropout         = 0.3,
    ):
        super().__init__()
        self.d_model = d_model

        # ── 1. CNN Backbone (ResNet50, fully frozen) ──────────────
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for p in resnet.parameters():
            p.requires_grad = False          # freeze ALL layers
        # Remove the FC head → output: (B*T, 2048, 1, 1)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])

        # ── 2. Input Projection: 2048 → d_model ──────────────────
        # Transforms CNN output dimension to Transformer working dimension
        self.input_proj = nn.Sequential(
            nn.Linear(2048, d_model),
            nn.LayerNorm(d_model),           # normalize before Transformer
            nn.Dropout(dropout)
        )

        # ── 3. Positional Encoding ────────────────────────────────
        # Injects frame order (0..15) into the sequence
        self.pos_encoding = PositionalEncoding(
            d_model  = d_model,
            max_len  = 16,
            dropout  = dropout
        )

        # ── 4. Transformer Encoder ────────────────────────────────
        # Each layer has:
        #   - Multi-Head Self-Attention (nhead heads)
        #   - Feed-Forward Network (d_model → dim_feedforward → d_model)
        #   - LayerNorm + Residual connections
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = nhead,
            dim_feedforward = dim_feedforward,
            dropout         = dropout,
            batch_first     = True,    # input shape: (batch, seq, features)
            norm_first      = True,    # Pre-LN: more stable training
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers = num_layers,
        )

        # ── 5. Classifier Head ────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        """
        x: (batch, frames, C, H, W)
        """
        B, T, C, H, W = x.shape

        # ── Step 1: CNN feature extraction per frame ──────────────
        # Merge batch and time → treat all frames as independent images
        x = x.view(B * T, C, H, W)              # (B*T, 3, 224, 224)

        with torch.no_grad():                    # CNN is frozen — no gradients needed
            cnn_feat = self.cnn(x)               # (B*T, 2048, 1, 1)

        cnn_feat = cnn_feat.view(B * T, -1)      # (B*T, 2048)  flatten spatial
        cnn_feat = cnn_feat.view(B, T, -1)       # (B, T, 2048) restore sequence

        # ── Step 2: Project 2048 → d_model ───────────────────────
        x = self.input_proj(cnn_feat)            # (B, T, 512)

        # ── Step 3: Add positional encoding ──────────────────────
        # Tells the Transformer which frame is first, second, etc.
        x = self.pos_encoding(x)                 # (B, T, 512)

        # ── Step 4: Temporal Transformer ─────────────────────────
        # All 16 frames processed in parallel
        # Each frame attends to every other frame
        x = self.transformer(x)                  # (B, T, 512)

        # ── Step 5: Mean pooling across frames ───────────────────
        # Average all 16 frame representations
        # Unlike LSTM (takes last state), Transformer treats all frames equally
        x = x.mean(dim=1)                        # (B, 512)

        # ── Step 6: Classification ────────────────────────────────
        return self.classifier(x)                # (B, num_classes)


# Initialise and inspect
model     = CNN_Transformer(num_classes=NUM_CLASSES).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'CNN+Transformer ready!')
print(f'Trainable params : {trainable:,}')
print(f'Total params     : {total:,}')
print(f'Frozen (CNN)     : {total - trainable:,}')

## 7. What happens inside the Transformer — visualized

Run this cell to understand the shape at each step.

In [ ]:
# Trace shapes through the model with a dummy batch
model.eval()
dummy = torch.randn(2, 16, 3, 224, 224).to(device)  # 2 videos, 16 frames

with torch.no_grad():
    B, T, C, H, W = dummy.shape
    print(f'Input                  : {list(dummy.shape)}')

    # CNN
    flat = dummy.view(B*T, C, H, W)
    cnn_out = model.cnn(flat)
    print(f'After CNN (per frame)  : {list(cnn_out.shape)}')

    cnn_out = cnn_out.view(B, T, -1)
    print(f'Reshaped to sequence   : {list(cnn_out.shape)}')

    # Projection
    proj = model.input_proj(cnn_out)
    print(f'After projection       : {list(proj.shape)}')

    # Positional encoding
    pos = model.pos_encoding(proj)
    print(f'After pos encoding     : {list(pos.shape)}')

    # Transformer
    trans_out = model.transformer(pos)
    print(f'After Transformer      : {list(trans_out.shape)}')

    # Mean pool
    pooled = trans_out.mean(dim=1)
    print(f'After mean pooling     : {list(pooled.shape)}')

    # Classifier
    out = model.classifier(pooled)
    print(f'Final output           : {list(out.shape)}')

print('\nShape trace complete!')

## 8. Train

In [ ]:
criterion  = nn.CrossEntropyLoss()

# Only Transformer + projection + classifier params are trained
# CNN params are frozen (requires_grad=False) so Adam ignores them
optimizer  = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4      # L2 regularization — helps prevent overfitting
)

# Cosine annealing — smoothly reduces LR to near zero over training
# Better than StepLR for Transformers (avoids sudden LR drops)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

history     = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
ckpt_path   = f'{CKPT_DIR}/cnn_transformer_best.pth'

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print('-' * 44)
    t0 = time.time()

    for phase in ['train', 'val']:
        model.train() if phase == 'train' else model.eval()
        loader = train_loader if phase == 'train' else val_loader

        running_loss, running_correct = 0.0, 0

        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs  = model(inputs)
                loss     = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                if phase == 'train':
                    loss.backward()
                    # Gradient clipping — prevents exploding gradients
                    # (Transformers are sensitive to large gradient updates)
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(), max_norm=1.0
                    )
                    optimizer.step()

            running_loss    += loss.item() * inputs.size(0)
            running_correct += torch.sum(preds == labels)

        if phase == 'train':
            scheduler.step()

        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc  = running_correct.double() / len(loader.dataset)

        history[f'{phase}_loss'].append(epoch_loss)
        history[f'{phase}_acc'].append(epoch_acc.item())
        print(f'  {phase.upper():5} → Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}')

        if phase == 'val' and epoch_acc > best_val_acc:
            best_val_acc = epoch_acc
            torch.save(model.state_dict(), ckpt_path)
            print(f'  ✅ Best saved! Val Acc: {best_val_acc:.4f}')

    print(f'  ⏱  {time.time()-t0:.1f}s')

print(f'\nTraining complete! Best Val Acc: {best_val_acc:.4f}')

## 9. Evaluate

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        outputs  = model(inputs.to(device))
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## 10. Plots — Accuracy, Loss, Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 5))

axes[0].plot(history['train_acc'],  label='Train', marker='o', color='steelblue')
axes[0].plot(history['val_acc'],    label='Val',   marker='o', color='darkorange')
axes[0].set_title('CNN+Transformer Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_loss'], label='Train', marker='o', color='steelblue')
axes[1].plot(history['val_loss'],   label='Val',   marker='o', color='darkorange')
axes[1].set_title('CNN+Transformer Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[2])
axes[2].set_title('Confusion Matrix', fontweight='bold')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
plt.setp(axes[2].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/cnn_transformer_results.png', dpi=150)
plt.show()
print('Results saved!')

## 11. Attention Weight Visualization
Shows which frames the Transformer attended to most for each video.
High attention on a frame = that frame was most informative for the prediction.

In [ ]:
import numpy as np

# Hook to capture attention weights from the first Transformer layer
attention_weights = []

def get_attention_hook(module, input, output):
    # TransformerEncoderLayer output is just the hidden states
    # We use register_forward_hook on self_attn instead
    pass


def get_avg_attention_for_batch(model, inputs):
    """
    Extract attention weights by hooking into the first
    Transformer layer's self-attention module.
    Returns average attention weight per frame position.
    """
    attn_output = []

    def hook_fn(module, input, output):
        # output[1] = attention weights when need_weights=True
        # Shape: (batch, nhead, seq, seq) or (batch, seq, seq)
        if isinstance(output, tuple) and output[1] is not None:
            attn_output.append(output[1].detach().cpu())

    # Register hook on first layer's self_attn
    hook = model.transformer.layers[0].self_attn.register_forward_hook(hook_fn)

    model.eval()
    with torch.no_grad():
        _ = model(inputs)

    hook.remove()

    if attn_output:
        # attn shape: (batch, seq, seq) — average over query positions
        attn = attn_output[0]              # (batch, 16, 16)
        avg  = attn.mean(dim=1)            # (batch, 16) — avg over queries
        return avg.mean(dim=0).numpy()     # (16,) — avg over batch
    return None


# Get one batch from val_loader
sample_inputs, sample_labels = next(iter(val_loader))
sample_inputs = sample_inputs.to(device)

avg_attn = get_avg_attention_for_batch(model, sample_inputs)

if avg_attn is not None:
    fig, ax = plt.subplots(figsize=(10, 3))
    frames  = list(range(1, 17))
    colors  = ['#2B5BA8' if a >= avg_attn.mean() else '#AEB6BF' for a in avg_attn]

    bars = ax.bar(frames, avg_attn, color=colors)
    ax.axhline(avg_attn.mean(), color='darkorange', linestyle='--',
               label=f'Mean ({avg_attn.mean():.3f})')
    ax.set_title(
        'Average Attention Weight per Frame\n'
        '(Blue = above average attention — Transformer found these frames most informative)',
        fontweight='bold'
    )
    ax.set_xlabel('Frame Index')
    ax.set_ylabel('Attention Weight')
    ax.set_xticks(frames)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/transformer_attention_weights.png', dpi=150)
    plt.show()
    print('\nAttention weight per frame:')
    for i, w in enumerate(avg_attn):
        bar = '█' * int(w * 200)
        print(f'  Frame {i+1:2d}: {w:.4f}  {bar}')
else:
    print('Attention weights not available — try with need_weights=True in self_attn')

## 12. LSTM vs Transformer — Final Comparison

In [ ]:
# Update lstm_acc with your actual result from 04_video_cnn_lstm.ipynb
lstm_acc        = 0.9962   # paste your CNN+LSTM best val acc here
transformer_acc = best_val_acc.item()

print(f'\n{"="*50}')
print(f'{"MODEL":<28} {"VAL ACCURACY":>18}')
print(f'{"="*50}')
print(f'{"CNN + LSTM":<28} {lstm_acc*100:>17.2f}%')
print(f'{"CNN + Transformer":<28} {transformer_acc*100:>17.2f}%')
print(f'{"="*50}')
diff = (transformer_acc - lstm_acc) * 100
sign = '+' if diff >= 0 else ''
print(f'Difference: {sign}{diff:.2f}%')
print()

# Architecture comparison table
print(f'{"COMPARISON":<30} {"LSTM":>10} {"TRANSFORMER":>14}')
print('-' * 56)
rows = [
    ('Frame processing',   'Sequential',   'Parallel'),
    ('Long-range frames',  'Degrades',     'Direct attention'),
    ('Gradient clipping',  'Not needed',   'Required'),
    ('Scheduler',          'StepLR',       'CosineAnnealing'),
    ('Temporal summary',   'Last state',   'Mean pooling'),
    ('Interpretable?',     'No',           'Attention weights'),
]
for label, lstm_val, trans_val in rows:
    print(f'{label:<30} {lstm_val:>10} {trans_val:>14}')